<a href="https://colab.research.google.com/github/amitpatelmsba/financial-intelligence-system/blob/main/RAG_Financial_Intelligence_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 RAG-Based Financial Intelligence System

This notebook implements a complete **Retrieval-Augmented Generation (RAG)** pipeline for financial document analysis.

### What this system does:
- 📄 **Ingests** financial documents (PDFs: earnings reports, SEC filings, 10-Ks, etc.)
- 🔍 **Embeds** and indexes document chunks into a ChromaDB vector store
- 🤖 **Retrieves** relevant context for user queries using semantic search
- 💡 **Generates** intelligent, cited answers using Claude (Anthropic)

### Sections:
1. Install Dependencies
2. Set Anthropic API Key
3. Document Processor
4. Vector Store (ChromaDB + SentenceTransformers)
5. RAG Engine (Claude)
6. Advanced RAG (Multi-query)
7. Upload Documents
8. Initialize & Index
9. Query System
10. Interactive Interface
11. Batch Export

## Step 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install anthropic langchain langchain-community chromadb pypdf sentence-transformers numpy pandas python-dotenv ipywidgets -q
print("✅ All dependencies installed!")

## Step 2: Set Your Anthropic API Key

Add your key in **Colab Secrets** (🔑 icon in left sidebar) with name `ANTHROPIC_API_KEY`, or enter it manually below.

In [ ]:
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Add ANTHROPIC_API_KEY in Colab sidebar -> Secrets (🔑 icon)
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Option 2: Enter manually (prompted securely)
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
    print("✅ API key set manually")

## Step 3: Document Processor

Handles loading PDFs and splitting them into overlapping chunks for embedding.

In [ ]:
# ─── Document Processor ────────────────────────────────────────────────────────
from pathlib import Path
from typing import List, Dict
import PyPDF2
from langchain.text_splitter import RecursiveCharacterTextSplitter

class DocumentProcessor:
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
            length_function=len,
        )

    def load_pdf(self, file_path: str) -> str:
        """Extract text from PDF"""
        text = ""
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text() or ""
        return text

    def chunk_text(self, text: str, metadata: Dict = None) -> List[Dict]:
        """Split text into chunks with metadata"""
        chunks = self.text_splitter.split_text(text)
        return [
            {"text": chunk, "metadata": metadata or {}, "chunk_id": i}
            for i, chunk in enumerate(chunks)
        ]

    def process_directory(self, directory: str) -> List[Dict]:
        """Process all PDFs in a directory"""
        all_chunks = []
        pdf_files = list(Path(directory).glob("*.pdf"))
        if not pdf_files:
            print(f"⚠️  No PDF files found in '{directory}'")
            return []
        for file_path in pdf_files:
            print(f"📄 Processing: {file_path.name}")
            text = self.load_pdf(str(file_path))
            chunks = self.chunk_text(
                text,
                metadata={"source": str(file_path), "filename": file_path.name}
            )
            all_chunks.extend(chunks)
            print(f"   → {len(chunks)} chunks extracted")
        return all_chunks

print("✅ DocumentProcessor class defined")

## Step 4: Vector Store

Uses **ChromaDB** for persistent vector storage and **SentenceTransformers** (`all-MiniLM-L6-v2`) for generating embeddings.

In [ ]:
# ─── Vector Store ──────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import chromadb
from typing import List, Dict
import uuid

class VectorStore:
    def __init__(self, collection_name: str = "financial_docs", persist_dir: str = "./chroma_db"):
        print("🔄 Loading embedding model (all-MiniLM-L6-v2)...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        print("✅ Embedding model loaded")

        self.client = chromadb.PersistentClient(path=persist_dir)
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        print(f"✅ Vector store ready — collection: '{collection_name}'")

    def add_documents(self, chunks: List[Dict]):
        """Add document chunks to vector store"""
        if not chunks:
            print("⚠️  No chunks to add.")
            return
        texts  = [chunk["text"] for chunk in chunks]
        metas  = [chunk["metadata"] for chunk in chunks]
        ids    = [str(uuid.uuid4()) for _ in chunks]

        print(f"🔄 Generating embeddings for {len(chunks)} chunks...")
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True).tolist()
        self.collection.add(embeddings=embeddings, documents=texts, metadatas=metas, ids=ids)
        print(f"✅ {len(chunks)} chunks added to vector store")

    def search(self, query: str, n_results: int = 5) -> Dict:
        """Semantic search for relevant documents"""
        total = self.collection.count()
        if total == 0:
            return {"documents": [], "metadatas": [], "distances": []}
        query_emb = self.embedding_model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_emb,
            n_results=min(n_results, total)
        )
        return {
            "documents": results["documents"][0] if results["documents"] else [],
            "metadatas": results["metadatas"][0] if results["metadatas"] else [],
            "distances": results["distances"][0] if results["distances"] else []
        }

    def count(self):
        return self.collection.count()

print("✅ VectorStore class defined")

## Step 5: RAG Engine

Connects ChromaDB retrieval with **Claude** (Anthropic) to generate grounded, source-cited answers.

In [ ]:
# ─── RAG Engine ────────────────────────────────────────────────────────────────
import anthropic, os

class RAGEngine:
    def __init__(self, vector_store: VectorStore):
        self.vector_store = vector_store
        self.client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
        self.model = "claude-opus-4-5"  # or "claude-3-5-sonnet-20241022"

    def retrieve_context(self, query: str, top_k: int = 5) -> str:
        """Retrieve relevant context from vector store"""
        results = self.vector_store.search(query, n_results=top_k)
        context_parts = []
        for i, (doc, meta, dist) in enumerate(zip(
            results["documents"], results["metadatas"], results["distances"]
        )):
            source    = meta.get("filename", "Unknown")
            relevance = round((1 - dist) * 100, 1)
            context_parts.append(f"[Source {i+1}: {source} | Relevance: {relevance}%]\n{doc}\n")
        return "\n".join(context_parts)

    def generate_response(self, query: str, top_k: int = 5) -> dict:
        """Generate a grounded response using RAG"""
        context = self.retrieve_context(query, top_k)

        if not context.strip():
            return {
                "answer": "No relevant documents found. Please upload financial documents first.",
                "context_used": "", "model": self.model
            }

        prompt = f"""You are a financial intelligence assistant. Use the provided context to answer the question accurately.

Context from financial documents:
{context}

User Question: {query}

Provide a detailed answer based on the context. Cite sources using [Source N]. If the context is insufficient, acknowledge this clearly."""

        msg = self.client.messages.create(
            model=self.model, max_tokens=2000,
            messages=[{"role": "user", "content": prompt}]
        )
        return {"answer": msg.content[0].text, "context_used": context, "model": self.model}

print("✅ RAGEngine class defined")

## Step 6: Advanced RAG Features

**Multi-query retrieval**: generates 3 query variations to broaden search coverage, then deduplicates and synthesizes results for higher recall.

In [ ]:
# ─── Advanced RAG Features ─────────────────────────────────────────────────────

class AdvancedRAG:
    """Enhanced RAG with multi-query retrieval"""

    def __init__(self, vector_store: VectorStore, rag_engine: RAGEngine):
        self.vector_store = vector_store
        self.rag_engine   = rag_engine

    def multi_query_retrieval(self, query: str) -> str:
        """Generate multiple query variations for better retrieval coverage"""
        variations_prompt = f"""Generate 3 different ways to phrase this financial question.
Return only the 3 variations, one per line, no numbering or extra text.
Original: {query}"""

        msg = self.rag_engine.client.messages.create(
            model=self.rag_engine.model, max_tokens=200,
            messages=[{"role": "user", "content": variations_prompt}]
        )
        variations = msg.content[0].text.strip().split("\n")

        all_docs = []
        for v in [query] + variations:
            if v.strip():
                results = self.vector_store.search(v.strip(), n_results=3)
                all_docs.extend(results["documents"])

        # Deduplicate while preserving order
        unique_docs = list(dict.fromkeys(all_docs))
        return "\n\n---\n\n".join(unique_docs[:10])

    def answer_with_multi_query(self, query: str) -> str:
        """Full pipeline: multi-query retrieval → generation"""
        print("🔄 Generating query variations for broader retrieval...")
        context = self.multi_query_retrieval(query)
        if not context.strip():
            return "No relevant documents found."

        prompt = f"""You are a financial intelligence assistant. Answer the following question based on the provided context.

Context:
{context}

Question: {query}

Provide a comprehensive, well-structured answer."""

        msg = self.rag_engine.client.messages.create(
            model=self.rag_engine.model, max_tokens=2000,
            messages=[{"role": "user", "content": prompt}]
        )
        return msg.content[0].text

print("✅ AdvancedRAG class defined")

## Step 7: Upload Financial Documents

Upload your PDF files (earnings reports, SEC 10-K/10-Q filings, annual reports, research notes, etc.).

> 💡 **Tip:** Click "Choose Files" and select multiple PDFs at once. If you have no PDFs, the system will auto-load demo financial data in Step 8.

In [ ]:
# ─── Upload & Process Documents ────────────────────────────────────────────────
import os
from google.colab import files

os.makedirs("/content/financial_docs", exist_ok=True)

print(" Upload your financial PDF documents")
print("   (Earnings reports, SEC 10-K/10-Q filings, annual reports, etc.)")
print("   Tip: Hold Cmd/Ctrl to select multiple files.\n")

uploaded = files.upload()

if uploaded:
    for filename, content in uploaded.items():
        filepath = f"/content/financial_docs/{filename}"
        with open(filepath, 'wb') as f:
            f.write(content)
        print(f" Saved: {filename} ({len(content)/1024:.1f} KB)")
    print(f"\n📁 Total files uploaded: {len(uploaded)}")
else:
    print(" No files uploaded — demo mode will be activated in Step 8.")

## Step 8: Initialize System & Index Documents

Initializes all components and builds the vector index. If no PDFs were uploaded, loads built-in demo data.

In [ ]:
# ─── Initialize System ─────────────────────────────────────────────────────────
import os
from pathlib import Path

# Initialize all components
processor    = DocumentProcessor(chunk_size=1000, chunk_overlap=200)
vector_store = VectorStore(collection_name="financial_intelligence", persist_dir="/content/chroma_db")
rag_engine   = RAGEngine(vector_store)
advanced_rag = AdvancedRAG(vector_store, rag_engine)

# Check for uploaded PDFs
pdf_dir   = "/content/financial_docs"
pdf_files = list(Path(pdf_dir).glob("*.pdf"))

if pdf_files:
    print(f"\n📂 Found {len(pdf_files)} PDF file(s). Processing...")
    chunks = processor.process_directory(pdf_dir)
    if chunks:
        vector_store.add_documents(chunks)
        print(f"\n System ready! {vector_store.count()} chunks indexed.")
    else:
        print("  No text could be extracted from the PDFs.")
else:
    # Demo mode: curated financial sample data
    print("\n No PDFs found — loading built-in demo financial data...\n")
    demo_chunks = [
        {
            "text": "Apple Inc. Q4 FY2024 Results: Revenue $94.9B (+6% YoY). iPhone revenue $46.2B. Services revenue record $24.2B (+12% YoY). Mac $7.7B (+2%). iPad $7.0B (+8%). Wearables $9.0B (-3%). Operating cash flow $26.8B. EPS $1.64 vs $1.46 consensus estimate.",
            "metadata": {"source": "demo", "filename": "AAPL_Q4_FY2024_Earnings.pdf"}, "chunk_id": 0
        },
        {
            "text": "Apple Risk Factors (10-K 2024): (1) Intense smartphone competition from Samsung, Google Pixel. (2) iPhone revenue concentration ~50% of total. (3) Supply chain dependency in China/Taiwan (TSMC). (4) EU Digital Markets Act App Store scrutiny. (5) Strong USD creating FX headwinds in international markets.",
            "metadata": {"source": "demo", "filename": "AAPL_10K_2024.pdf"}, "chunk_id": 1
        },
        {
            "text": "Microsoft Q1 FY2025 Earnings: Total revenue $65.6B (+16% YoY). Intelligent Cloud $24.1B (+20%), Azure grew 33%. Productivity & Business Processes $28.3B (+12%). More Personal Computing $13.2B (+17%). Net income $24.7B. EPS $3.30 vs $3.10 estimate. AI products contributed ~$3B incremental revenue.",
            "metadata": {"source": "demo", "filename": "MSFT_Q1_FY2025_Earnings.pdf"}, "chunk_id": 2
        },
        {
            "text": "Federal Reserve Monetary Policy Update 2024: Fed Funds Rate maintained at 5.25%-5.50% — highest since 2001. CPI inflation 2.4%, PCE 2.1%, trending toward 2% target. Financial institutions face net interest margin compression. 2-year/10-year yield curve inverted for 18+ months. Next rate cut probability 65% in Q1 2025.",
            "metadata": {"source": "demo", "filename": "Fed_Monetary_Policy_2024.pdf"}, "chunk_id": 3
        },
        {
            "text": "NVIDIA Q3 FY2025 Record Earnings: Revenue $35.1B (+94% YoY, +17% QoQ). Data Center $30.8B driven by H100/H200/B100 GPU demand from hyperscalers. Gaming $3.3B (+15%). Gross margin expanded to 74.6%. Q4 guidance $37.5B ±2%. NVIDIA controls ~80% of AI training accelerator market. Blackwell platform ramp accelerating.",
            "metadata": {"source": "demo", "filename": "NVDA_Q3_FY2025_Earnings.pdf"}, "chunk_id": 4
        },
        {
            "text": "S&P 500 Sector Performance 2024 YTD: Technology +45% (AI-driven, led by NVDA +230%, AAPL +18%). Healthcare +8% (GLP-1 drugs tailwind). Energy -3% (oil weakness, WTI avg $78). Financials +18% (higher-for-longer rates benefit). Consumer Discretionary +12% (AMZN, TSLA recovery). Industrials +15% (infrastructure spend). Materials -2%.",
            "metadata": {"source": "demo", "filename": "SP500_Sector_Report_2024.pdf"}, "chunk_id": 5
        },
        {
            "text": "Goldman Sachs 2025 Market Outlook: S&P 500 target 6,200 (+8% from current). GDP growth 2.4%. Core inflation 2.0% by mid-2025. Fed to cut 3x (75bps total). Key risks: escalating trade tensions, commercial real estate stress, election uncertainty. Overweight: Tech, Healthcare, Industrials. Underweight: Utilities, Materials.",
            "metadata": {"source": "demo", "filename": "GS_Market_Outlook_2025.pdf"}, "chunk_id": 6
        },
    ]
    vector_store.add_documents(demo_chunks)
    print(f"\n Demo mode ready! {vector_store.count()} financial data chunks indexed.")

## Step 9: Query the System

Ask financial questions and get grounded answers from your indexed documents.

In [ ]:
# ─── Query Helper ──────────────────────────────────────────────────────────────

def ask(question: str, use_advanced: bool = False):
    """Query the RAG system and display results clearly"""
    print("\n" + "="*72)
    print(f"❓ QUESTION: {question}")
    print("="*72)

    if use_advanced:
        answer = advanced_rag.answer_with_multi_query(question)
    else:
        result = rag_engine.generate_response(question)
        answer = result["answer"]

    print("\n ANSWER:")
    print("-"*72)
    print(answer)
    print("="*72 + "\n")
    return answer


# ── Sample Queries ───────────────────────────────────────────────────────────
example_questions = [
    "What were the revenue trends in the latest quarter?",
    "What are the main risk factors mentioned in the documents?",
    "How is the AI and cloud sector performing?",
    "What is the market outlook for 2025?",
]

for q in example_questions:
    ask(q)

## Step 10: Interactive Query Interface

Type any financial question and click **Ask Question** to get an answer in real time.

In [ ]:
# ─── Interactive Widget Interface ──────────────────────────────────────────────
from ipywidgets import widgets
from IPython.display import display, clear_output

query_input = widgets.Textarea(
    value='What were the key financial metrics reported?',
    placeholder='Ask a financial question about your documents...',
    description='Question:',
    layout=widgets.Layout(width='100%', height='80px')
)

advanced_toggle = widgets.Checkbox(
    value=False,
    description='Use Advanced RAG (multi-query expansion for better recall)',
)

submit_btn = widgets.Button(
    description='  Ask Question',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='170px', height='36px')
)

output_area = widgets.Output()

def on_submit(b):
    with output_area:
        clear_output()
        question = query_input.value.strip()
        if question:
            ask(question, use_advanced=advanced_toggle.value)
        else:
            print("  Please enter a question first.")

submit_btn.on_click(on_submit)

print("🎛️  RAG Financial Intelligence — Interactive Interface")
print("─" * 55)
display(query_input, advanced_toggle, submit_btn, output_area)

## Step 11: Batch Query & Export Results

Run multiple questions at once and download a CSV with all Q&A pairs.

In [ ]:
# ─── Batch Query & CSV Export ──────────────────────────────────────────────────
import pandas as pd
from datetime import datetime
from google.colab import files

# ── Define your batch questions here ────────────────────────────────────────
batch_questions = [
    "What was the total revenue reported?",
    "What are the main risk factors?",
    "How did companies perform compared to the prior year?",
    "What is the guidance or outlook for next quarter?",
    "What were the key cost drivers or expenses?",
    "How is the AI and cloud sector performing?",
    "What does the market outlook say for 2025?",
]

print(" Running batch queries...\n")
results_data = []

for i, question in enumerate(batch_questions, 1):
    print(f"[{i}/{len(batch_questions)}] {question[:65]}...")
    try:
        result = rag_engine.generate_response(question, top_k=3)
        results_data.append({
            "question":  question,
            "answer":    result["answer"],
            "model":     result["model"],
            "timestamp": datetime.now().isoformat()
        })
    except Exception as e:
        results_data.append({
            "question":  question,
            "answer":    f"Error: {str(e)}",
            "model":     "N/A",
            "timestamp": datetime.now().isoformat()
        })

# Save results
df = pd.DataFrame(results_data)
output_path = "/content/rag_financial_results.csv"
df.to_csv(output_path, index=False)
print(f"\n Results saved to {output_path}")

# Preview
print("\n📊 RESULTS PREVIEW:")
print("="*72)
for r in results_data:
    print(f"\nQ: {r['question']}")
    print(f"A: {r['answer'][:280]}{'...' if len(r['answer']) > 280 else ''}")
    print("-"*60)

# Download the CSV
files.download(output_path)
print("\n CSV downloaded!")